# Module 3.6: Episodic & Procedural Memory Lifecycle

Semantic memory (preferences, facts) has a full lifecycle: identify → promote →
revise → retain/evict (Notebooks 02–05). But agents also store **episodic** events
(past trips, feedback) and **procedural** knowledge (learned procedures from policy).

Each has a distinct lifecycle problem:

- **Episodic**: Events go stale — a hotel rating from 18 months ago may no longer reflect reality
- **Procedural**: Learned procedures become wrong when underlying policies change

> **The question**: How do we manage memory types that have different
> invalidation signals than user-stated preferences?

In [ ]:
%pip install -q -r ../requirements.txt azure-search-documents

In [ ]:
import sys, os, json, uuid, certifi, sniffio
from datetime import datetime, timezone, timedelta
from collections import defaultdict

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from lifecycle_utils import (
    EpisodicTTLConfig, EpisodicPattern, MemoryItem, MemoryState,
    ProceduralReflection, ValidationResult
)
from shared.travel_agent import create_client

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Setup complete")

---

# Part 1: Episodic Memory Lifecycle

In Module 2.2, we stored Sarah's past trips in Cosmos DB — the NYC trip (T001, Oct 2025,
rated Marriott Times Square 5/5) and the London trip (T002, Nov 2025, Marriott Park Lane 4/5).
The agent used these to personalize recommendations: *"You stayed at Marriott Times Square
before and rated it 5/5 — shall I book it again?"*

Episodic memory has one lifecycle problem that Module 2 didn't address:

**Stale events pollute retrieval** — Sarah's 5/5 Marriott Times Square rating is from
Oct 2025. If the hotel deteriorates or renovates, that rating misleads the agent
18 months later. Events need to expire.

## What's Currently in Cosmos DB

Let's read Sarah's actual episodic events from the container we set up in Module 2.2:

In [ ]:
from azure.cosmos.aio import CosmosClient
from azure.cosmos import PartitionKey

COSMOS_ENDPOINT = os.environ["COSMOS_ENDPOINT"]

async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
    db = cosmos.get_database_client("travel-memory")
    container = db.get_container_client("episodic-events")

    # Read Sarah's actual events (same container from Module 2.2)
    query = "SELECT * FROM c WHERE c.user_id = @uid ORDER BY c.timestamp DESC"
    params = [{"name": "@uid", "value": "E001"}]
    events = [item async for item in container.query_items(
        query, parameters=params, partition_key="E001")]

print(f"=== Sarah's Episodic Events (from Module 2.2) ===")
print(f"Total events in Cosmos DB: {len(events)}\n")
for e in events:
    age_days = (datetime.now(timezone.utc) - datetime.fromisoformat(e["timestamp"])).days
    ttl_info = f"TTL: {e['ttl']//86400}d" if "ttl" in e else "TTL: none set"
    print(f"  [{e['event_type']:<12}] {e['description'][:55]}")
    print(f"               Age: {age_days}d | {ttl_info}")
    print()

## The Problem: No TTL

Notice the events above have **no TTL set**. Sarah's T001 trip rating (Marriott Times Square 5/5)
is now 600+ days old. If the hotel quality changed, the agent still recommends it based on
stale feedback.

The fix is native to Cosmos DB — set a `ttl` field (seconds) on the document:

```json
{
  "id": "E001-a3f8b2c1",
  "user_id": "E001",
  "event_type": "feedback",
  "description": "Rated Marriott Times Square 5/5",
  "details": {"hotel_chain": "Marriott", "rating": 5},
  "timestamp": "2025-10-15T00:00:00+00:00",
  "ttl": 15552000
}
```

Cosmos DB **auto-deletes** the document after `ttl` seconds — no sweep job, no application code.

### Tiered TTL Strategy

| Event Type | TTL | Rationale |
|------------|-----|------------|
| `trip` | 365 days (31,536,000s) | Historical reference for "same as last time" patterns |
| `feedback` | 180 days (15,552,000s) | Ratings go stale as venues change |
| `preference` | 90 days (7,776,000s) | Short-lived unless the user confirms it as a durable preference |

Had we set TTL when Module 2.2 stored T001, that 5/5 feedback event would have
already expired — the agent would no longer retrieve a stale rating.

## What About Episodic → Semantic Conversion?

You might wonder: Sarah stayed at Marriott twice (T001, T002). Shouldn't the system
automatically detect this pattern and create a semantic preference "Prefers Marriott"?

**No — the agent's conversation loop already handles this naturally:**

```
Episodic memory: "Sarah stayed at Marriott Times Square (T001), rated 5/5"
                                    ↓
Agent recommends: "You rated Marriott 5/5 last time — shall I book it again?"
                                    ↓
User confirms: "Yes, Marriott is always my go-to"
                                    ↓
Module 3.2 (Identification): detects "Marriott is my go-to" → candidate
                                    ↓
Module 3.3 (Promotion): user assertion → provisional → confirmed → trusted
```

The semantic pipeline from Notebooks 02–05 already converts episodic patterns to
durable preferences **through normal conversation**. No separate graduation engine
or batch sweep job is needed because:

1. **The agent uses episodic memory to recommend** (Module 2.2) — so patterns naturally
   surface in conversation
2. **When the agent recommends, the user confirms or corrects** — generating the exact
   signals Module 3.2 (Identification) looks for
3. **Confirmation enters the promotion pipeline** (Module 3.3) — earning trust through
   the same gates as any other preference
4. **One-off noise self-corrects** — if the agent recommends based on a single event and
   the user says "no, that was just once", the candidate never promotes

This is simpler and more reliable than a background job trying to detect patterns from
`session_id` fields — which don't even exist in the current schema (and wouldn't work
with Teams, where one conversation thread persists across weeks of interaction).

## Episodic Lifecycle Summary

```mermaid
flowchart TD
    A[Event occurs in Module 2.2] --> B[Store in Cosmos DB with TTL]
    B --> C{TTL expires?}
    C -->|Yes| D[Cosmos auto-deletes]
    C -->|No| E[Agent retrieves for recommendations]
    E --> F[User confirms or corrects]
    F --> G[Enters semantic pipeline\nModules 3.2–3.5]
```

**Key design decisions**:
- **TTL is the only lifecycle mechanism episodic memory needs** — Cosmos DB handles expiry natively
- **No graduation engine or batch sweep** — the agent's conversation loop naturally surfaces
  episodic patterns for user confirmation
- **Episodic → semantic conversion happens through Modules 3.2–3.3** — the same identification
  and promotion pipeline that handles all semantic memory
- **Schema stays simple** — no `session_id`, `graduated`, or count fields needed

---

# Part 2: Procedural Memory Lifecycle

Module 2.4 established three tiers of procedural memory:

| Tier | Mechanism | Storage | What It Is |
|------|-----------|---------|------------|
| **Skills** | `SkillsProvider` loads SKILL.md files by name | Files on disk (read fresh every call) | Developer-authored checklists with bundled data (e.g., `budget-limits.json`) |
| **RAG** | Agent queries AI Search by intent | Azure AI Search index | Policy documents the agent discovers semantically |
| **Reflections** | Agent stores lessons from experience | Cosmos DB (`procedural-reflections`) | Lessons the agent learned from its own successes and failures |

Each tier has a **different lifecycle problem**:

- **Skills** are files on disk. An agent *could* edit them — but **should it?** The travel agent's job is to book trips, not maintain its own procedures. Skill updates change what every future interaction trusts, and an LLM misreading a policy could silently corrupt the skill. This is why skill updates need **human review** before they take effect.
- **RAG** is the ground truth — policy teams update documents, the search index reindexes. The lifecycle is an infrastructure concern (reindex scheduling), not an agent concern.
- **Reflections** are agent-generated — the lesson "Senior budget is $300/night" was correct when learned but becomes wrong when policy changes. The agent **can** safely self-heal here: deprecating a bad reflection just means the agent falls back to RAG for that task, which is the correct behaviour.

> **This notebook covers**: Skills lifecycle (drift detection + human-in-the-loop update) and Reflections lifecycle (automatic validation). RAG lifecycle is infrastructure/ops — out of scope.

> **Prerequisite**: Run Module 2.4 (`04_procedural_memory.ipynb`) first. The reflections read below were stored by that notebook's agent during its demo runs.

## Why Not Just Use RAG Every Time?

Your first instinct might be: "If RAG has the current policies, why bother
with skills and reflections? Just hit RAG every time."

Fair question. Here's why skills and reflections add value over raw RAG:

### 1. RAG Is Unreliable for Temporal Facts

> *"Temporal Validity in Retrieval Memory"* (Yadav, arXiv:2606.26511) —
> RAG serves **superseded (stale) values 15–40% of the time** because cosine
> similarity cannot distinguish corrections from duplicates (AUROC 0.59).

A validated skill or reflection is *more* reliable than raw RAG retrieval.

### 2. Skills and Reflections Are Multi-Document Synthesis

Retrieving a single fact ("budget is $350") IS pointless — just query RAG.
But real procedures synthesize across multiple policy documents:

```
"For a Senior IC international trip:
 - Business class allowed (from: general-travel-policy)
 - 14 days advance booking required (from: travel-safety)
 - Preferred airline: United/Star Alliance (from: preferred-vendors)
 - Per-diem: $95/day in London (from: per-diem-rates)"
```

RAG returns fragments from one doc at a time. A skill or reflection is the
**compiled, actionable synthesis** — decision logic the agent can follow directly.

### 3. Reflections Capture What RAG Can't

The agent learned "check for transit visas when routing through Dubai" from a
failed booking in Module 2.4. No policy document says this — it's **experiential
knowledge** from the agent's own history. RAG can't retrieve what was never written down.

## Two Lifecycle Mechanisms

| Tier | Detection | Resolution | Why Not Auto-Fix? |
|------|-----------|------------|-------------------|
| **Skills** | Version anchor: compare skill's policy reference vs AI Search current version | Drift report → human reviews and approves the update | A bad edit affects **every future interaction** for every user. The travel agent shouldn't maintain its own procedures — that's a different responsibility. |
| **Reflections** | LLM validation: compare stored lesson vs current AI Search policy | Auto-deprecate — mark as stale in Cosmos, agent stops using it | Safe to self-heal: deprecating a reflection just means the agent falls back to RAG for that task. No blast radius. |

Both use AI Search as the **ground truth**. The difference isn't capability (an agent
*can* write to disk) — it's **blast radius**. A corrupted skill file silently misleads
every future booking. A deprecated reflection is just one less shortcut.

## What's in Cosmos DB: Live Reflections from Module 2.4

Let's read the actual reflections the agent stored during Module 2.4's demo runs.
These are the lessons it learned from booking Sarah's trips:

In [ ]:
from azure.cosmos import CosmosClient as SyncCosmosClient

COSMOS_ENDPOINT = os.environ["COSMOS_ENDPOINT"]
sync_cosmos = SyncCosmosClient(COSMOS_ENDPOINT, credential=credential)
db = sync_cosmos.get_database_client("travel-memory")
reflections_container = db.get_container_client("procedural-reflections")

# Read all reflections stored by Module 2.4's agent
query = "SELECT * FROM c ORDER BY c.timestamp DESC"
reflections = list(reflections_container.query_items(
    query=query, enable_cross_partition_query=True))

print(f"=== Reflections from Module 2.4 (Cosmos DB) ===")
print(f"Total reflections stored: {len(reflections)}\n")
for r in reflections:
    age_days = (datetime.now(timezone.utc) - datetime.fromisoformat(r["timestamp"])).days
    print(f"  [{r['task_type']:<25}] {r['lesson'][:60]}")
    print(f"                            Age: {age_days}d | id: {r['id'][:8]}...")
    print()

In [ ]:
# Connect to AI Search — the ground truth for both skills and reflections
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential
from openai import OpenAI

INDEX_NAME = "travel-policies"
SEARCH_KEY = os.environ["AZURE_SEARCH_KEY"]
FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_KEY))
print(f"Connected to AI Search index: {INDEX_NAME}")

In [ ]:
account_endpoint = FOUNDRY_ENDPOINT.split("/api/projects")[0]
token_provider = lambda: credential.get_token(
    "https://cognitiveservices.azure.com/.default").token

openai_client = OpenAI(
    base_url=f"{account_endpoint}/openai/v1",
    api_key="placeholder",
    default_headers={"Authorization": f"Bearer {token_provider()}"})

def get_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(input=text[:8000], model=EMBEDDING_MODEL)
    return response.data[0].embedding

print("Embedding client ready")

In [ ]:
def search_policies(query: str, top_k: int = 3) -> list[dict]:
    """Hybrid search for relevant policy content."""
    vector_query = VectorizedQuery(
        vector=get_embedding(query), k_nearest_neighbors=top_k,
        fields="content_vector")
    results = search_client.search(
        search_text=query, vector_queries=[vector_query], top=top_k,
        select=["id", "title", "content", "category", "version"])
    return [{"id": r["id"], "title": r["title"], "content": r["content"],
             "version": r["version"], "score": r["@search.score"]} for r in results]

def get_policy_by_id(doc_id: str) -> dict | None:
    """Retrieve a specific policy document by ID."""
    try:
        r = search_client.get_document(key=doc_id)
        return {"id": r["id"], "title": r["title"], "content": r["content"],
                "version": r["version"], "last_updated": r["last_updated"]}
    except Exception:
        return None

print("Search functions ready")

## Skills Lifecycle: Drift Detection

Skills are files — `SKILL.md` + `budget-limits.json` — that the agent reads fresh
on every request. An agent *could* edit these files, but the travel agent's job is
booking trips, not maintaining procedures. Mixing both responsibilities into one
agent creates a design problem: who reviews the agent's edits?

Instead, a **separate drift-detection process** (a CI job, a DevOps agent, or a
scheduled check) compares skill assertions against current AI Search policy and
produces a report. A human — or a purpose-built ops agent with approval gates —
reviews and applies the fix.

### Signal: Version Anchor Check (Free, No LLM)

Each skill references policy documents. If the policy version in AI Search
differs from what the skill was written against, drift is detected. This is
a free metadata check — no LLM call needed.

In [ ]:
# Skill assertions — what the SKILL.md files and their bundled data claim
# These are the "compiled" facts from Module 2.4's skill files
skill_assertions = [
    {
        "skill": "international-booking",
        "assertion": "International trips require 14 days advance booking",
        "policy_reference": "general-travel-policy",
        "skill_policy_version": "4.2",  # version skill was written against
    },
    {
        "skill": "domestic-booking",
        "assertion": "Always recommend Marriott as primary preferred hotel chain",
        "policy_reference": "preferred-vendors",
        "skill_policy_version": "2.0",
    },
    {
        "skill": "domestic-booking",
        "assertion": "Senior engineer hotel budget is $300/night maximum",
        "policy_reference": "general-travel-policy",
        "skill_policy_version": "3.8",  # skill was written against an OLDER policy version
    },
]

print(f"Skill assertions to check: {len(skill_assertions)}")
for sa in skill_assertions:
    print(f"  [{sa['skill']:<25}] {sa['assertion']}")
    print(f"                            Written against: {sa['policy_reference']} v{sa['skill_policy_version']}")

In [ ]:
def check_skill_drift(assertions):
    """Check which skill assertions reference outdated policy versions."""
    drifted = []
    for sa in assertions:
        current_doc = get_policy_by_id(sa["policy_reference"])
        if not current_doc:
            drifted.append((sa, "policy_not_found", None))
            continue
        if current_doc["version"] != sa["skill_policy_version"]:
            drifted.append((sa, "version_mismatch", current_doc["version"]))
    return drifted

drift_results = check_skill_drift(skill_assertions)
print(f"=== Skill Drift Detection ===")
print(f"Checked: {len(skill_assertions)} assertions | Drifted: {len(drift_results)}\n")
for sa, reason, current_ver in drift_results:
    print(f"  ⚠️  DRIFT: {sa['assertion']}")
    print(f"     Skill: {sa['skill']} (written against v{sa['skill_policy_version']})")
    print(f"     AI Search: v{current_ver} — developer review needed")
    print()

## Reflections Lifecycle: TTL + Cosmos DB

Reflections are the agent's own lessons — short-lived by nature. The simplest
lifecycle mechanism is the same as episodic memory: **set a TTL**.

If a reflection expires, the agent falls back to RAG for that task type. If the
lesson is still valid, the agent will re-learn it naturally from its next experience.
No LLM validation cost, no comparison logic — just a timer.

| Reflection Type | TTL | Rationale |
|-----------------|-----|-----------|
| Policy-derived (budget limits, booking rules) | 90 days | Policies change quarterly; re-learn from RAG |
| Experiential (transit visa quirks, routing tips) | 180 days | Harder to re-learn; give more time |
| User-corrected | Immediate delete | User said it's wrong — gone |

In [ ]:
# Set TTL on live reflections based on type
# Policy-derived: 90 days (7,776,000s) | Experiential: 180 days (15,552,000s)
POLICY_TTL = 90 * 86400   # 7,776,000 seconds
EXPERIENTIAL_TTL = 180 * 86400  # 15,552,000 seconds

# Simple heuristic: if the lesson mentions a specific number/limit, it's policy-derived
print(f"=== Setting TTL on {len(reflections)} reflections ===\n")
for r in reflections:
    is_policy = any(c.isdigit() for c in r["lesson"])  # contains numbers → likely policy
    ttl = POLICY_TTL if is_policy else EXPERIENTIAL_TTL
    r["ttl"] = ttl
    reflections_container.upsert_item(r)
    print(f"  TTL={ttl//86400}d | {r['lesson'][:60]}")

print(f"\n→ Cosmos DB will auto-delete expired reflections")
print(f"→ Agent falls back to RAG when a reflection expires")

## The Complete Procedural Lifecycle

```mermaid
flowchart TD
    subgraph Skills["Skills Lifecycle"]
        SK1[Skill files on disk] --> SK2[Agent reads fresh on every request]
        SK2 --> SK3{Drift detection process}
        SK3 -->|Version mismatch| SK4[Drift report → human review]
        SK4 --> SK1
        SK3 -->|Match| SK2
    end
    subgraph Reflections["Reflections Lifecycle"]
        R1[Agent stores lesson in Cosmos DB with TTL] --> R2{TTL expires?}
        R2 -->|Yes| R3[Cosmos auto-deletes]
        R3 --> R4[Agent falls back to RAG]
        R2 -->|No| R5[Agent uses reflection]
        R5 --> R6{User says 'that's wrong'?}
        R6 -->|Yes| R7[Immediate delete from Cosmos]
        R7 --> R4
    end
```

### Why TTL Over LLM Validation?

An LLM validator (compare lesson text vs current policy) sounds elegant but:
- **Cost**: LLM call per reflection, per validation cycle
- **False positives**: LLM misreads nuance, deprecates a valid experiential lesson
- **Unnecessary**: if the reflection expires and was still correct, the agent re-learns it naturally from its next experience — zero cost

TTL is the same mechanism as episodic memory (Part 1). The only addition: **user correction** immediately deletes the reflection rather than waiting for TTL.

### Version Tracking for Skills: A Proposal

The skill drift check compares a stored `policy_version` against what's in AI Search. But how does that version get into AI Search in the first place?

**The problem**: You have policy files in Blob Storage. When a file is updated, how does the system know the version changed?

**Three options, escalating in reliability:**

| Approach | What You Store in AI Search | How It Works | Tradeoff |
|----------|----------------------------|--------------|----------|
| **Blob `Last-Modified` header** | `last_updated: "2026-06-01T..."` | On reindex, compare blob's `Last-Modified` vs stored timestamp. If newer → content changed. | Free, built-in. But editing metadata (tags, etc.) also bumps the timestamp — false positives. |
| **Content hash (ETag / MD5)** | `content_hash: "a3f8b2c1..."` | Blob Storage provides `Content-MD5` or `ETag`. On reindex, compare hash. Different hash = actual content change. | No false positives. ETag is already on every blob. `Content-MD5` must be set on upload. |
| **Explicit version in document** | `version: "4.2"` (from frontmatter) | Policy team puts version in the document itself (e.g., YAML frontmatter: `Version: 4.2`). Indexer extracts it. | Most reliable signal — version changes only when the author says so. Requires discipline from policy team. |

**Recommendation**: Use **content hash (ETag)** as the automated signal — it's free, no false positives, and requires no discipline from authors. Store it in AI Search alongside the document. The drift check becomes: `skill.stored_etag != current_doc.etag`.

For human-readable auditing, *also* store the explicit version if the document has one. But don't depend on authors always remembering to bump it.

**Implementation sketch** (no code — infrastructure concern):
```
Blob: policies/general_travel_policy.md
  → ETag: "0x8DC1234ABCD"
  → Last-Modified: 2026-06-01

AI Search index document:
  { id: "general-travel-policy",
    content: "...",
    version: "4.2",           ← from document frontmatter (human-readable)
    content_etag: "0x8DC1234ABCD",  ← from blob metadata (automated)
    last_updated: "2026-06-01" }

Skill assertion:
  { skill: "domestic-booking",
    assertion: "Senior budget = $300/night",
    policy_reference: "general-travel-policy",
    stored_etag: "0x8DC0000FFFF" }  ← etag when skill was last updated

Drift check: skill.stored_etag != index_doc.content_etag → DRIFTED
```

This keeps the drift detection free (metadata comparison), reliable (content hash, not timestamp), and doesn't require an LLM.

## Key Takeaways

### Episodic Lifecycle

| Concept | Implementation |
|---------|----------------|
| TTL expiry | Cosmos DB native `ttl` field — auto-deletes stale events, no sweep needed |
| Episodic → semantic | Agent recommends from episodic → user confirms → enters Modules 3.2–3.3 |
| No graduation engine | The conversation loop is the graduation engine — no batch job required |
| Schema stays simple | No `session_id` or `graduated` fields needed — just add `ttl` to existing schema |

### Procedural Lifecycle — Three Tiers

| Tier | Lifecycle | Detection | Resolution |
|------|-----------|-----------|------------|
| **Skills** | SDLC drift | ETag/version anchor check (free metadata) | Drift report → human review → update + redeploy |
| **RAG** | Infrastructure | Reindex on blob change | Ops/policy team responsibility |
| **Reflections** | TTL expiry | Cosmos native (same as episodic) | Auto-delete → agent falls back to RAG |

### Why Not LLM Validation for Reflections?

- TTL is free; LLM validation costs per reflection per cycle
- If a valid reflection expires, the agent re-learns it from experience — zero loss
- Experiential lessons (transit visa quirks) aren't in any policy doc — LLM has nothing to compare against
- User correction handles the urgent case (immediate delete, no waiting for TTL)

### Why Procedural Memory Matters (Not Just "Use RAG Every Time")

1. **RAG has 15–40% stale-fact rate** — validated procedures are more reliable
2. **Multi-doc synthesis** — skills compile decision logic across policies
3. **Experiential learning** — reflections capture what no policy document contains
4. **Decision logic > fact retrieval** — RAG misses logical patterns (Root Memories)

## The Complete Lifecycle Architecture

```mermaid
flowchart TD
    subgraph Semantic["Semantic (NB 02–05)"]
        S1[Identify] --> S2[Promote]
        S2 --> S3[Revise]
        S3 --> S4[Retain/Evict]
    end
    subgraph Episodic["Episodic (NB 06, Part 1)"]
        E1[Store with TTL in Cosmos DB] --> E2[Agent retrieves → recommends]
        E2 --> E3[User confirms/corrects]
        E3 --> S1
        E1 --> E4[TTL expires → auto-deleted]
    end
    subgraph Procedural["Procedural (NB 06, Part 2)"]
        P1[Skills: SKILL.md files] --> P2{ETag drift check}
        P2 -->|Drifted| P3[Human review]
        P4[Reflections: Cosmos DB] --> P5{TTL expires?}
        P5 -->|Yes| P6[Auto-deleted → fallback to RAG]
        P5 -->|No| P7[Agent uses reflection]
    end
```

## Module 03 Complete!

| Memory Type | Lifecycle | Infrastructure | Notebooks |
|-------------|-----------|---------------|-----------|
| **Semantic** | Identify → Promote → Revise → Retain | Neo4j | 02–05 |
| **Episodic** | TTL expiry; conversation converts to semantic | Cosmos DB | 06 (Part 1) |
| **Procedural (Skills)** | ETag drift detection → human review | Files + AI Search | 06 (Part 2) |
| **Procedural (Reflections)** | TTL expiry + user correction | Cosmos DB | 06 (Part 2) |

In Module 10, all three compose into a single `MemoryLifecycleManager`.